In [78]:
# Combine all data from P1, P2 into final CCRI layer

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [141]:
import os
import pandas as pd
import geopandas as gpd
import glob
import numpy as np

In [142]:
# Query data dir (avoiding hard-coding paths when working between users)
#/content/drive/MyDrive/CCRI/ccri_repo/data
data_dir = '/content/drive/MyDrive/CCRI/CCRR_repo/data'
print(data_dir)

/content/drive/MyDrive/CCRI/CCRR_repo/data


In [82]:
# Check that data dir was input correctly
data_dir

'/content/drive/MyDrive/CCRI/CCRR_repo/data'

In [83]:
# Read in all the data we need
# -- exposure files
p1_exposure_file = pd.read_csv('{}/misc/Merged_Exposure_Data.csv'.format(data_dir))
# Identify columns to rescale (exclude 'iso3' or non-numeric columns)
cols_to_rescale = p1_exposure_file.select_dtypes(include='number').columns
# Apply min-max scaling to 0–10
p1_exposure_file[cols_to_rescale] = p1_exposure_file[cols_to_rescale].apply(
    lambda col: 10 * (col - col.min()) / (col.max() - col.min())
)


p2_exposure_file = pd.read_csv('{}/misc/P2_Merged_Normalized_avg.csv'.format(data_dir))
p1p2_scores = pd.read_csv('{}/misc/p1_p2_avg_ccri.csv'.format(data_dir))

# -- attribute files
wb_income = pd.read_csv('{}/misc/WB_INCOME.csv'.format(data_dir))
unicef_ro = pd.read_csv('{}/misc/UNICEF_PROG_REG_GLOBAL.csv'.format(data_dir))

# -- population files
childpop = pd.read_csv('{}/misc/child_pop_sum_adm0.csv'.format(data_dir))
#worldpop = pd.read_csv('{}/Misc/World_Population_ByAOI_adm0.csv'.format(data_dir))

# -- boundary file
adm0 = gpd.read_file('{}/misc/adm0_boundaries_simple.geojson'.format(data_dir))

# -- fragile codes
fragile = pd.read_csv('{}/misc/List of fragile context (2025).csv'.format(data_dir))

# -- component vals
p1_components = pd.read_csv('{}/misc/p1_group_mean.csv'.format(data_dir))
# Identify numeric columns to rescale (exclude non-numeric like 'iso3', 'group', etc.)
cols_to_rescale = p1_components.select_dtypes(include='number').columns
# Apply min-max scaling to range 0–10
p1_components[cols_to_rescale] = p1_components[cols_to_rescale].apply(
    lambda col: 10 * (col - col.min()) / (col.max() - col.min())
)

p2_components = pd.read_csv('{}/misc/p2_group_mean.csv'.format(data_dir))

In [84]:
# List of columns to exclude from missing value calculation
exclude_cols = ['iso3', 'P2_arithmetic_avg', 'rank_reverse']

# Subset of columns to calculate missingness on
cols_to_check = [col for col in p2_exposure_file.columns if col not in exclude_cols]

# Calculate missing percentage per row (country)
p2_exposure_file['P2_missing_val'] = p2_exposure_file[cols_to_check].isna().mean(axis=1) * 100

# Preview the result
print(p2_exposure_file[['iso3', 'P2_missing_val']].head())


  iso3  P2_missing_val
0  AFG       11.764706
1  ALB       11.764706
2  DZA       11.764706
3  AND       58.823529
4  AGO        5.882353


In [85]:
# Renaming some columns
p1_exposure_file.columns = [col.replace('_absolute', '_abs_norm') if '_absolute' in col else col for col in p1_exposure_file.columns]
p1_exposure_file.columns = [col.replace('_relative', '_rel_norm') if '_relative' in col else col for col in p1_exposure_file.columns]

In [86]:
# Merge P exposures by ISO3
merged_P = (p1_exposure_file.merge(p2_exposure_file, on='iso3', how='left'))
all_P = (merged_P.merge(p1p2_scores, on='iso3', how='left'))

In [87]:
all_P = all_P.drop(columns=['P2_arithmetic_avg_y', 'rank_reverse_x'])
all_P = all_P.rename(columns={'P2_arithmetic_avg_x': 'P2_arithmetic_avg', 'rank_reverse_y': 'rank_reverse'})

In [88]:
# Add WB income
wb_income = wb_income[['Region_Code', 'ISO3Code']]
wb_income.loc[:, 'Region_Code'] = wb_income['Region_Code'].str.extract(r'WB_(.*)')
df = (all_P.merge(wb_income, left_on='iso3', right_on='ISO3Code', how='left').drop('ISO3Code', axis=1).rename(columns={'Region_Code': 'wb_income'}))

In [89]:
# Add Regional Office
unicef_ro = unicef_ro[['Region_Code','ISO3Code']]
unicef_ro.loc[:,'Region_Code'] = unicef_ro['Region_Code'].str.extract(r'UNICEF_(.*)')
df = (df.merge(unicef_ro, left_on='iso3', right_on='ISO3Code', how='left').drop('ISO3Code', axis=1).rename(columns={'Region_Code': 'unicef_ro'}))

In [90]:
# Add population data
# -- Take the childpop data from the geojson of p1p2 avg
gdf = gpd.read_file('{}/misc/p1_p2_avg_ccri.geojson'.format(data_dir))
df_grouped = gdf[['ISO3', 'child_population_total', 'population_total']].groupby('ISO3', as_index=False).mean()
df_w_childpop = (df.merge(df_grouped, left_on=['iso3'], right_on=['ISO3'], how='left').rename(columns={'child_population_total': 'u18_pop'})).drop(columns=['ISO3'])

#df_w_childpop = (df.merge(childpop, left_on='iso3', right_on='ISO3', how='left').rename(columns={'child_population': 'u18_pop'})).drop(columns=['child_population_gpw'])
#df_w_childpop['u18_pop'] = df_w_childpop['u18_pop'].astype(int)

In [91]:
df_w_childpop = df_w_childpop.rename(columns={'iso3':'ISO3'})

In [94]:
# Using simplified boundaries for geometry
adm0 = adm0[['ISO3', 'name', 'ucode','uuid','geometry','type']]
adm0_unique = adm0.drop_duplicates(subset=['ISO3'], keep='first')
df_combined = (df_w_childpop.merge(adm0_unique, left_on=['ISO3'], right_on=['ISO3'], how='left'))

In [96]:
# Grabbing actual exposure numbers
# Define file paths
exposure_path = "{}/pillar1_data".format(data_dir)

# Get all CSV files for exposure
exposure_files = glob.glob(os.path.join(exposure_path, "*.csv"))

# Initialize empty list for processed data
exposure_data_list = []

ZERO_FILL_FILES = [
    "agricultural_drought_fao_1984-2023_exposure_adm0.csv",
    "tropical_storm_100yr_giri_2024_exposure_adm0.csv",
    "vectorborne_malariapf_2012-2022_exposure_adm0.csv",
    "vectorborne_malariapv_2012-2022_exposure_adm0.csv"
]

### **Process Each File in One Loop**
for file in exposure_files:
    df = pd.read_csv(file)  # Read full file to check available columns
    df = df.rename(columns={
        'iso3': 'ISO3'
    })
    df = (df.groupby('ISO3').sum(numeric_only=True, min_count=1).reset_index())

    filename_only = os.path.basename(file)
    hazard_name = '_'.join(filename_only.split('_')[:2])  # Extract hazard name

    if filename_only in ZERO_FILL_FILES:
        # Fill missing numeric values with 0 for specified hazards
        df['child_population_exposed'] = df.get('child_population_exposed', pd.Series(dtype=float)).fillna(0)

    # Ensure required columns exist
    required_cols = {'ISO3', 'child_population_exposed'}
    if not required_cols.issubset(df.columns):
        print(f"Skipping {file}: Missing columns {required_cols - set(df.columns)}")
        continue  # Skip if required columns are missing

    df.dropna(subset=['child_population_exposed'], inplace=True)

    # Compute relative exposure (%)
    df['{}_rel'.format(hazard_name)] = np.where(
        (df['child_population_total'] > 0) & (~df['child_population_total'].isna()),
        (df['child_population_exposed'] / df['child_population_total']) * 100,
        0
    )

    # Rename to hazard
    df = df.rename(columns={'child_population_exposed': '{}_abs'.format(hazard_name)})
    df = df.drop(columns=['system:index','child_population_total', 'population_total'])

    df_combined = (df_combined.merge(df, left_on=['ISO3'], right_on=['ISO3'], how='left'))

In [97]:
df_combined = df_combined.rename(columns={'population_total': 'total_pop'})

In [98]:
#add exposures by hazards topics
topics_path = '/content/drive/MyDrive/p1_exposure'

# Get all CSV files for exposure
topic_files = glob.glob(os.path.join(topics_path, "*topic*.csv"))
for file in topic_files:
    topic_name = file.split('/')[-1].replace('_exposure_adm0.csv', '')
    print(topic_name)
    df = pd.read_csv(file)  # Read full file to check available columns
    df = df.rename(columns={
        'iso3': 'ISO3'
    })
    df = (df.groupby('ISO3').sum(numeric_only=True, min_count=1).reset_index())
    df = df.rename(columns={'child_population_exposed': '{}_abs'.format(topic_name)})
    df['{}_rel'.format(topic_name)] = df['{}_abs'.format(topic_name)]/df['child_population_total']*100
    df = df[['ISO3',f'{topic_name}_abs',f'{topic_name}_rel']]
    df_combined = (df_combined.merge(df, left_on=['ISO3'], right_on=['ISO3'], how='left'))


ge_1_topic
ge_2_topic
ge_3_topic
ge_4_topic
ge_5_topic
ge_6_topic
ge_7_topic
ge_8_topic
drought_topic
heatwave_topic
fire_topic


In [99]:
quad_table = pd.read_csv(os.path.join(data_dir, 'misc', 'ccri_quadrant_table.csv'))
quad_table = quad_table.rename(columns={'Quadrant': 'CCRI_Quadrant'})

In [100]:
df_combined = df_combined.merge(quad_table, on=['ISO3'], how='left')

In [105]:
multi_hazards_files = glob.glob(os.path.join(data_dir, 'misc', 'Multi_Hazard_intensity_Exposure_adm0_TH_Percentile*.csv'))

all_hazards = None

for file in multi_hazards_files:
    suffix = file.split('/')[-1].replace('Multi_Hazard_intensity_Exposure_adm0_TH_Percentile', '').replace('.csv', '')
    abs_col = f'mhi_TH{suffix}_abs'
    rel_col = f'mhi_TH{suffix}_rel'

    df = pd.read_csv(file, usecols=['ISO3', 'child_population_exposed'], dtype={'child_population_exposed': float})
    df = df.groupby('ISO3', as_index=False)['child_population_exposed'].sum()
    df = df.rename(columns={'child_population_exposed': abs_col})

    # # Clean ISO3
    # df = df.dropna(subset=['ISO3'])
    # df['ISO3'] = df['ISO3'].astype(str).str.strip()
    # df = df.drop_duplicates(subset='ISO3')

    # Merge with df_combined to compute relative exposure
    df = df.merge(df_combined[['ISO3', 'u18_pop']], on='ISO3', how='left')
    df[rel_col] = (df[abs_col] / df['u18_pop'])* 100
    df[rel_col] = df[rel_col].clip(upper=100)

    # Drop to avoid duplication
    df = df.drop(columns=['u18_pop'])

    if all_hazards is None:
        all_hazards = df
    else:
        all_hazards = all_hazards.merge(df, on='ISO3', how='outer')



In [106]:
df_combined = df_combined.merge(all_hazards, on=['ISO3'], how='left')

In [108]:
# --- Adding P2 data
vul_path = "{}/pillar2_data".format(data_dir)
total_population_file = "{}/pillar1_data/agricultural_drought_fao_1984-2023_exposure_adm0.csv".format(data_dir)

# Load total child population data
total_pop_df = pd.read_csv(total_population_file, usecols=['iso3', 'adm0_name', 'child_population_total'])
total_pop_df = total_pop_df.rename(columns={'iso3': 'ISO3'})

# Ensure unique ISO3-name pairs before merging
total_pop_df = total_pop_df.groupby(['ISO3'], as_index=False).agg({'child_population_total': 'sum'})

# Get all CSV files for exposure
p2_vul_files = glob.glob(os.path.join(vul_path, "*.csv"))

# Initialize empty list for processed data
vul_data_list = []

for file in p2_vul_files:
    df = pd.read_csv(file)  # Read full file to check available columns
    df = df[df['time_period'] >= 2015]
    if 'iso3' not in df.columns or 'obs_value' not in df.columns:
        continue  # Skip files missing required columns
    # Normalize 'value' column
    df = df[['iso3', 'obs_value']].dropna()
    df = df.rename(columns={'iso3': 'ISO3'}, errors='ignore')
    filename_only = os.path.basename(file)
    hazard_name = '_'.join(filename_only.split('.csv')[:1])  # Extract hazard name
    if hazard_name == 'P2_Child_Mortality':
        continue

    # merge with population data
    df = (df.merge(total_pop_df, on='ISO3', how='left'))

    # rename to hazard and norm
    df['{}'.format(hazard_name)] = np.where(
        (df['child_population_total'] > 0) & (~df['child_population_total'].isna()),
        (df['obs_value']),
        0
    )

    # Rename for relative
    df = df.drop(columns=['child_population_total', 'time_period', 'data_source', 'obs_value'], errors='ignore')

    vul_data_list.append(df)

In [109]:
for i in range(len(vul_data_list)):
    df_combined = (df_combined.merge(vul_data_list[i], on=['ISO3'], how='left'))

In [110]:
# Add fragile
fragile['fragile'] = 'fragile'
df_combined = (df_combined.merge(fragile[['ISO3','fragile']], on=['ISO3'], how='left'))

In [113]:
# Add components
# Rename
p1_components = p1_components.rename(columns={'river_flood_gmean': 'P1_rfl', 'coastal_flood_gmean': 'P1_cfl',
                                             'storm_gmean': 'P1_ts', 'drought_gmean': 'P1_dr', 'heatwave_gmean': 'P1_hw', 'extreme_heat_gmean': 'P1_ext',
                                             'fire_gmean': 'P1_fr', 'sand_dust_gmean': 'P1_sds', 'air_pollution_gmean': 'P1_pm25',
                                             'malaria_gmean': 'P1_mal'})

p2_components = p2_components.rename(columns={'health': 'P2_hea', 'nutrition': 'P2_nut', 'education': 'P2_edu', 'wash' : 'P2_wash',
                                             'protection': 'P2_pro', 'poverty': 'P2_pov', 'survival': 'P2_sur'})

In [115]:
df_combined = (df_combined.merge(p1_components, left_on=['ISO3'], right_on=['iso3'], how='left'))

In [116]:
df_combined = (df_combined.merge(p2_components, left_on=['ISO3'], right_on=['iso3'], how='left'))

In [118]:
# Remove any columns we don't want and rearrange as well
# -- Dropping
# Drop min, max
df_combined = df_combined.drop(df_combined.filter(regex='max').columns, axis=1)
df_combined = df_combined.drop(df_combined.filter(regex='min').columns, axis=1)

# -- Renaming
df_combined = df_combined.rename(columns={'name': 'adm_name', 'P1_P2_geometric_avg': 'ccri', 'ISO3':'iso3'})

In [119]:
# Set to 2 decimal places
for col in df_combined.columns:
    if type(df_combined['{}'.format(col)].iloc[0]) != str:
        if col in ['wb_income', 'unicef_ro', 'geometry', 'fragile']:
            continue
        else:
            df_combined[col] = df_combined[col].round(2)

In [120]:
# Rename normalized to _norm
df_combined.columns = [col.replace('_normalized', '_norm') if 'normalized' in col else col for col in df_combined.columns]

In [121]:
# Renaming the hazards
# P1 hazards
df_combined.columns = [col.replace('river_flood', 'rfl') if 'river_flood' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('coastal_flood', 'cfl') if 'coastal_flood' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('tropical_storm', 'ts') if 'tropical_storm' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('agricultural_drought', 'agdr') if 'agricultural_drought' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('drought_spei', 'metdr_spei') if 'drought_spei' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('drought_spi', 'metdr_spi') if 'drought_spi' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('heatwave_frequency', 'hw_fre') if 'heatwave_frequency' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('heatwave_severity', 'hw_sev') if 'heatwave_severity' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('heatwave_duration', 'hw_dur') if 'heatwave_duration' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('extreme_heat', 'ext') if 'extreme_heat' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('fire_frequency', 'fr_fre') if 'fire_frequency' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('fire_FRP', 'fr_int') if 'fire_FRP' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('sand_dust', 'sds') if 'sand_dust' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('vectorborne_malariapv', 'mal_pv') if 'vectorborne_malariapv' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('vectorborne_malariapf', 'mal_pf') if 'vectorborne_malariapf' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('air_pollution', 'pm25') if 'air_pollution' in col else col for col in df_combined.columns]

# P2 hazards
df_combined.columns = [col.replace('P2_DTP1_access', 'hea_dtp1') if 'P2_DTP1_access' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_DTP3_access', 'hea_dtp3') if 'P2_DTP3_access' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Skilled_birth_coverage', 'hea_skat') if 'P2_Skilled_birth_coverage' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_electricity_access', 'hea_elec') if 'P2_electricity_access' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Stunting', 'nut_stu') if 'P2_Stunting' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Child_Food_Poverty', 'nut_fpov') if 'P2_Child_Food_Poverty' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_At-least_basic_drinking_water', 'wash_wat') if 'P2_At-least_basic_drinking_water' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_WASH_Drinking_Sanitation', 'wash_san') if 'P2_WASH_Drinking_Sanitation' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_At-least_basic_sanitation', 'wash_san') if 'P2_At-least_basic_sanitation' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Basic_hygiene', 'wash_hyg') if 'P2_Basic_hygiene' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Lower_secondary_out_of_school', 'edu_lsos') if 'P2_Lower_secondary_out_of_school' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Lower_secondary_completion_rate', 'edu_lscr') if 'P2_Lower_secondary_completion_rate' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Learning_poverty', 'edu_lpov') if 'P2_Learning_poverty' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Child_labor', 'pro_lab') if 'P2_Child_labor' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Child_marriage', 'pro_mar') if 'P2_Child_marriage' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Child_poverty', 'pov_md') if 'P2_Child_poverty' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Under_five_covered_by_social_protection', 'pov_u5sp') if 'P2_Under_five_covered_by_social_protection' in col else col for col in df_combined.columns]
df_combined.columns = [col.replace('P2_Under_five_mortality', 'sur_u5mor') if 'P2_Under_five_mortality' in col else col for col in df_combined.columns]


In [126]:
# Automated way to generate the same dictionary
rename_dict = {}
for i in range(1, 9):
    rename_dict[f'ge_{i}_topic_abs'] = f'mhc_ge{i}_abs'
    rename_dict[f'ge_{i}_topic_rel'] = f'mhc_ge{i}_rel'

df_combined = df_combined.rename(columns=rename_dict)

In [122]:
df_combined.columns = [col.replace('value_norm', 'norm') if 'value_norm' in col else col for col in df_combined.columns]

In [123]:
# Rename some things
df_combined = df_combined.rename(columns={'name': 'adm_name', 'ISO3':'iso3'})


In [128]:
df_combined = df_combined[[
    'iso3',
    'adm_name',
    'total_pop',
    'u18_pop',
    'wb_income',
    'unicef_ro',
    'ucode',
    'uuid',
    'geometry',
    'type',
    'fragile','rfl_abs',
 'rfl_rel','rfl_abs_norm',
 'rfl_rel_norm','cfl_abs',
 'cfl_rel','cfl_abs_norm',
 'cfl_rel_norm','ts_abs',
 'ts_rel','ts_abs_norm',
 'ts_rel_norm','agdr_abs',
 'agdr_rel','agdr_abs_norm',
 'agdr_rel_norm','metdr_spei_abs',
 'metdr_spei_rel','metdr_spei_abs_norm',
 'metdr_spei_rel_norm','metdr_spi_abs',
 'metdr_spi_rel','metdr_spi_abs_norm',
 'metdr_spi_rel_norm','hw_fre_abs',
 'hw_fre_rel','hw_fre_abs_norm',
 'hw_fre_rel_norm','hw_dur_abs',
 'hw_dur_rel','hw_dur_abs_norm',
 'hw_dur_rel_norm','hw_sev_abs',
 'hw_sev_rel','hw_sev_abs_norm',
 'hw_sev_rel_norm','ext_abs',
 'ext_rel','ext_abs_norm',
 'ext_rel_norm','fr_fre_abs',
 'fr_fre_rel','fr_fre_abs_norm',
 'fr_fre_rel_norm','fr_int_abs',
 'fr_int_rel','fr_int_abs_norm',
 'fr_int_rel_norm','sds_abs',
 'sds_rel','sds_abs_norm',
 'sds_rel_norm','pm25_abs',
 'pm25_rel','pm25_abs_norm',
 'pm25_rel_norm','mal_pv_abs',
 'mal_pv_rel','mal_pv_abs_norm',
 'mal_pv_rel_norm','mal_pf_abs',
 'mal_pf_rel','mal_pf_abs_norm',
 'mal_pf_rel_norm','hea_dtp1','hea_dtp1_norm', 'hea_dtp3','hea_dtp3_norm','hea_skat','hea_skat_norm',
'hea_elec','hea_elec_norm','nut_stu','nut_stu_norm','nut_fpov','nut_fpov_norm','wash_wat',
    'wash_wat_norm','wash_san','wash_san_norm','wash_hyg','wash_hyg_norm','edu_lsos','edu_lsos_norm',
    'edu_lscr','edu_lscr_norm','edu_lpov','edu_lpov_norm','pro_lab','pro_lab_norm','pro_mar',
    'pro_mar_norm','pov_md','pov_md_norm','pov_u5sp','pov_u5sp_norm','sur_u5mor','sur_u5mor_norm',
    'P1_rfl', 'P1_cfl', 'P1_ts', 'P1_dr', 'P1_hw','P1_ext', 'P1_fr', 'P1_sds', 'P1_pm25', 'P1_mal',
    'P2_hea', 'P2_nut','P2_wash', 'P2_edu', 'P2_pro', 'P2_pov', 'P2_sur','P1_geometric_avg','P2_arithmetic_avg', 'P2_missing_val',
    'mhi_TH75_abs', 'mhi_TH80_abs', 'mhi_TH85_abs', 'mhi_TH95_abs', 'mhi_TH90_abs',
    'mhc_ge1_abs', 'mhc_ge2_abs', 'mhc_ge3_abs', 'mhc_ge4_abs', 'mhc_ge5_abs',
    'mhc_ge6_abs', 'mhc_ge7_abs', 'mhc_ge8_abs', #'mhc_ge9_abs', 'mhc_ge10_abs',
    'mhi_TH75_rel', 'mhi_TH80_rel', 'mhi_TH85_rel', 'mhi_TH95_rel', 'mhi_TH90_rel',
    'mhc_ge1_rel', 'mhc_ge2_rel', 'mhc_ge3_rel', 'mhc_ge4_rel', 'mhc_ge5_rel',
    'mhc_ge6_rel', 'mhc_ge7_rel', 'mhc_ge8_rel', #'mhc_ge9_rel', 'mhc_ge10_rel',
    'drought_topic_abs','drought_topic_rel','heatwave_topic_abs','heatwave_topic_rel',
 'fire_topic_abs','fire_topic_rel',#'malaria_topic_abs','malaria_topic_rel',
'ccri','CCRI_Quadrant']]

In [129]:
# List of columns to cap
exposure_cols = [
    'mhi_TH75_abs', 'mhi_TH80_abs', 'mhi_TH85_abs', 'mhi_TH95_abs', 'mhi_TH90_abs',
    'mhc_ge1_abs', 'mhc_ge2_abs', 'mhc_ge3_abs', 'mhc_ge4_abs', 'mhc_ge5_abs',
    'mhc_ge6_abs', 'mhc_ge7_abs', 'mhc_ge8_abs'#, 'mhc_ge9_abs', 'mhc_ge10_abs'
]

# Cap each exposure column at u18_pop
for col in exposure_cols:
    df_combined.loc[:, col] = np.minimum(df_combined[col], df_combined['u18_pop'])


In [130]:
df_combined = df_combined[df_combined['iso3'] != 'NIC']

In [131]:
for col in df_combined.columns:
    if 'abs' in col and 'norm' not in col:
        # Fill NaNs with 0 (or another valid integer) before casting
        df_combined.loc[:, col] = df_combined[col].fillna(0).astype(int)


In [132]:
# remove rows with (ccri is null) OR (iso3 == 'PSE')
mask = df_combined['ccri'].isna() | (df_combined['iso3'].str.upper() == 'PSE')
df_combined = df_combined[~mask]


In [133]:
df_combined.loc[:,'total_pop'] = df_combined['total_pop'].round(0)
df_combined.loc[:,'u18_pop'] = df_combined['u18_pop'].round(0)
df_combined.loc[:,'total_pop'] = pd.to_numeric(df_combined['total_pop'], errors='coerce').astype('Int64')
df_combined.loc[:,'u18_pop'] = pd.to_numeric(df_combined['u18_pop'], errors='coerce').astype('Int64')

In [134]:
df_combined.drop('ccri', axis=1, inplace=True)

In [135]:
gdf = gpd.GeoDataFrame(df_combined, geometry=df_combined['geometry'], crs='EPSG:4326')

In [136]:
gdf.to_file('{}/misc/CCRI_P1_P2_format.geojson'.format(data_dir))

In [12]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from functools import reduce

# ==========================================
# 1. SETUP & UTILITY FUNCTIONS
# ==========================================
data_dir = '/content/drive/MyDrive/CCRI/CCRR_repo/data'
print(f"Working Directory: {data_dir}")

def min_max_scale(df):
    """Applies 0-10 min-max scaling to numeric columns."""
    cols = df.select_dtypes(include='number').columns
    if len(cols) > 0:
        df[cols] = df[cols].apply(lambda col: 10 * (col - col.min()) / (col.max() - col.min()) if (col.max() - col.min()) != 0 else 0)
    return df

HAZARD_MAP = {
    'river_flood': 'rfl', 'coastal_flood': 'cfl', 'tropical_storm': 'ts',
    'agricultural_drought': 'agdr', 'drought_spei': 'metdr_spei', 'drought_spi': 'metdr_spi',
    'heatwave_frequency': 'hw_fre', 'heatwave_severity': 'hw_sev', 'heatwave_duration': 'hw_dur',
    'extreme_heat': 'ext', 'fire_frequency': 'fr_fre', 'fire_FRP': 'fr_int',
    'sand_dust': 'sds', 'vectorborne_malariapv': 'mal_pv', 'vectorborne_malariapf': 'mal_pf',
    'air_pollution': 'pm25', 'P2_DTP1_access': 'hea_dtp1', 'P2_DTP3_access': 'hea_dtp3',
    'P2_Skilled_birth_coverage': 'hea_skat', 'P2_electricity_access': 'hea_elec',
    'P2_Stunting': 'nut_stu', 'P2_Child_Food_Poverty': 'nut_fpov',
    'P2_At-least_basic_drinking_water': 'wash_wat', 'P2_WASH_Drinking_Sanitation': 'wash_san',
    'P2_At-least_basic_sanitation': 'wash_san', 'P2_Basic_hygiene': 'wash_hyg',
    'P2_Lower_secondary_out_of_school': 'edu_lsos', 'P2_Lower_secondary_completion_rate': 'edu_lscr',
    'P2_Learning_poverty': 'edu_lpov', 'P2_Child_labor': 'pro_lab', 'P2_Child_marriage': 'pro_mar',
    'P2_Child_poverty': 'pov_md', 'P2_Under_five_covered_by_social_protection': 'pov_u5sp',
    'P2_Under_five_mortality': 'sur_u5mor', 'value_normalized': 'norm', 'value_norm': 'norm'
}

ZERO_FILL_FILES = [
    "agricultural_drought_fao_1984-2023_exposure_adm0.csv",
    "tropical_storm_100yr_giri_2024_exposure_adm0.csv",
    "vectorborne_malariapf_2012-2022_exposure_adm0.csv",
    "vectorborne_malariapv_2012-2022_exposure_adm0.csv"
]

# ==========================================
# 2. LOAD BASE DATA (THE MASTER LIST)
# ==========================================
print("Loading base attributes and geometries...")
adm0 = gpd.read_file(f'{data_dir}/misc/adm0_boundaries_simple.geojson')[['ISO3', 'name', 'ucode', 'uuid', 'geometry', 'type']]

# Topology fix for dissolve
adm0['geometry'] = adm0['geometry'].buffer(0)
df_master = adm0.dissolve(by='ISO3', aggfunc='first').reset_index().rename(columns={'name': 'adm_name'})

# Populations
gdf_avg = gpd.read_file(f'{data_dir}/misc/p1_p2_avg_ccri.geojson')
df_grouped = gdf_avg[['ISO3', 'child_population_total', 'population_total']].groupby('ISO3', as_index=False).sum(min_count=1)
df_grouped = df_grouped.rename(columns={'child_population_total': 'u18_pop', 'population_total': 'total_pop'})

# Attributes
wb_income = pd.read_csv(f'{data_dir}/misc/WB_INCOME.csv')[['Region_Code', 'ISO3Code']]
wb_income['Region_Code'] = wb_income['Region_Code'].str.replace('WB_', '')
wb_income = wb_income.rename(columns={'Region_Code': 'wb_income', 'ISO3Code': 'ISO3'})

unicef_ro = pd.read_csv(f'{data_dir}/misc/UNICEF_PROG_REG_GLOBAL.csv')[['Region_Code', 'ISO3Code']]
unicef_ro['Region_Code'] = unicef_ro['Region_Code'].str.replace('UNICEF_', '')
unicef_ro = unicef_ro.rename(columns={'Region_Code': 'unicef_ro', 'ISO3Code': 'ISO3'})

fragile = pd.read_csv(f'{data_dir}/misc/List of fragile context (2025).csv')[['ISO3']]
fragile['fragile'] = 'fragile'

quad_table = pd.read_csv(f'{data_dir}/misc/ccri_quadrant_table.csv').rename(columns={'Quadrant': 'CCRI_Quadrant'})

# Initial merges
dfs_to_merge = [df_grouped, wb_income, unicef_ro, fragile, quad_table]
for df_to_add in dfs_to_merge:
    df_master = pd.merge(df_master, df_to_add, on='ISO3', how='left')

# ==========================================
# 3. P1 & P2 SCORES / COMPONENTS
# ==========================================
print("Processing Pillar scores...")
p1_scores = min_max_scale(pd.read_csv(f'{data_dir}/misc/Merged_Exposure_Data.csv')).rename(columns={'iso3': 'ISO3'})
p1_scores.columns = p1_scores.columns.str.replace('_absolute', '_abs_norm').str.replace('_relative', '_rel_norm')

p2_scores = pd.read_csv(f'{data_dir}/misc/P2_Merged_Normalized_avg.csv').rename(columns={'iso3': 'ISO3'})
# Identify missing values before merging P2 averages
cols_to_check = [c for c in p2_scores.columns if c not in ['ISO3', 'P2_arithmetic_avg', 'rank_reverse']]
p2_scores['P2_missing_val'] = p2_scores[cols_to_check].isna().mean(axis=1) * 100
# Drop P2_arithmetic_avg here to avoid suffixes during merge, as we pull it from p1p2_avg instead
if 'P2_arithmetic_avg' in p2_scores.columns:
    p2_scores = p2_scores.drop(columns=['P2_arithmetic_avg'])

# Combined P1/P2/CCRI scores from the single average file (Source of Truth)
p1p2_avg = pd.read_csv(f'{data_dir}/misc/p1_p2_avg_ccri.csv').rename(columns={'iso3': 'ISO3'})
p1p2_avg = p1p2_avg[['ISO3', 'P1_geometric_avg', 'P2_arithmetic_avg', 'P1_P2_geometric_avg']].rename(columns={'P1_P2_geometric_avg': 'ccri'})

p1_comps = min_max_scale(pd.read_csv(f'{data_dir}/misc/p1_group_mean.csv')).rename(columns={
    'iso3': 'ISO3', 'river_flood_gmean': 'P1_rfl', 'coastal_flood_gmean': 'P1_cfl',
    'storm_gmean': 'P1_ts', 'drought_gmean': 'P1_dr', 'heatwave_gmean': 'P1_hw',
    'extreme_heat_gmean': 'P1_ext', 'fire_gmean': 'P1_fr', 'sand_dust_gmean': 'P1_sds',
    'air_pollution_gmean': 'P1_pm25', 'malaria_gmean': 'P1_mal'
})

p2_comps = pd.read_csv(f'{data_dir}/misc/p2_group_mean.csv').rename(columns={
    'iso3': 'ISO3', 'health': 'P2_hea', 'nutrition': 'P2_nut', 'education': 'P2_edu',
    'wash' : 'P2_wash', 'protection': 'P2_pro', 'poverty': 'P2_pov', 'survival': 'P2_sur'
})

score_dfs = [p1_scores, p2_scores, p1p2_avg, p1_comps, p2_comps]
for df_to_add in score_dfs:
    df_master = pd.merge(df_master, df_to_add, on='ISO3', how='left')

# ==========================================
# 4. LOAD & PROCESS EXPOSURES
# ==========================================
print("Processing Hazard Exposures...")

for file in glob.glob(f"{data_dir}/pillar1_data/*.csv"):
    df = pd.read_csv(file).rename(columns={'iso3': 'ISO3'})
    df = df.groupby('ISO3').sum(numeric_only=True, min_count=1).reset_index()
    filename = os.path.basename(file)
    hazard = '_'.join(filename.split('_')[:2])

    if filename in ZERO_FILL_FILES:
        df['child_population_exposed'] = df.get('child_population_exposed', pd.Series(dtype=float)).fillna(0)

    if 'child_population_exposed' in df.columns:
        df[f'{hazard}_rel'] = np.where((df['child_population_total'] > 0) & df['child_population_total'].notna(),
                                       (df['child_population_exposed'] / df['child_population_total']) * 100, 0)
        df = df.rename(columns={'child_population_exposed': f'{hazard}_abs'})
        df_master = pd.merge(df_master, df[['ISO3', f'{hazard}_abs', f'{hazard}_rel']], on='ISO3', how='left')

for file in glob.glob('/content/drive/MyDrive/p1_exposure/*topic*.csv'):
    topic = os.path.basename(file).replace('_exposure_adm0.csv', '')
    df = pd.read_csv(file).rename(columns={'iso3': 'ISO3'})
    df = df.groupby('ISO3').sum(numeric_only=True, min_count=1).reset_index()
    df[f'{topic}_abs'] = df['child_population_exposed']
    df[f'{topic}_rel'] = np.where((df['child_population_total'] > 0) & df['child_population_total'].notna(),
                                  (df['child_population_exposed'] / df['child_population_total']) * 100, 0)
    df_master = pd.merge(df_master, df[['ISO3', f'{topic}_abs', f'{topic}_rel']], on='ISO3', how='left')

for file in glob.glob(f"{data_dir}/misc/Multi_Hazard_intensity_Exposure_adm0_TH_Percentile*.csv"):
    suffix = os.path.basename(file).replace('Multi_Hazard_intensity_Exposure_adm0_TH_Percentile', '').replace('.csv', '')
    abs_col, rel_col = f'mhi_TH{suffix}_abs', f'mhi_TH{suffix}_rel'
    df = pd.read_csv(file, usecols=['ISO3', 'child_population_exposed']).rename(columns={'child_population_exposed': abs_col})
    df = df.groupby('ISO3', as_index=False)[abs_col].sum(min_count=1)

    temp = pd.merge(df, df_master[['ISO3', 'u18_pop']], on='ISO3', how='left')
    temp[rel_col] = np.where((temp['u18_pop'] > 0) & temp['u18_pop'].notna(),
                             (temp[abs_col] / temp['u18_pop'] * 100), 0)
    temp[rel_col] = temp[rel_col].clip(upper=100)
    df_master = pd.merge(df_master, temp[['ISO3', abs_col, rel_col]], on='ISO3', how='left')

# ==========================================
# 5. LOAD & PROCESS VULNERABILITY (P2)
# ==========================================
print("Processing P2 Vulnerabilities...")
for file in glob.glob(f"{data_dir}/pillar2_data/*.csv"):
    df = pd.read_csv(file).rename(columns={'iso3': 'ISO3'})
    if 'ISO3' not in df.columns or 'obs_value' not in df.columns: continue
    if 'time_period' in df.columns:
        df = df[df['time_period'] >= 2015].sort_values('time_period')
        df = df.drop_duplicates(subset=['ISO3'], keep='last')

    hazard = os.path.basename(file).split('.csv')[0]
    if hazard == 'P2_Child_Mortality': continue

    df = df[['ISO3', 'obs_value']].rename(columns={'obs_value': hazard})
    df_master = pd.merge(df_master, df, on='ISO3', how='left')

# ==========================================
# 6. POST-PROCESSING & CLEANUP
# ==========================================
print("Final clean up and formatting...")

for old_name, new_name in HAZARD_MAP.items():
    df_master.columns = df_master.columns.str.replace(old_name, new_name)

rename_ge = {f'ge_{i}_topic_abs': f'mhc_ge{i}_abs' for i in range(1, 9)}
rename_ge.update({f'ge_{i}_topic_rel': f'mhc_ge{i}_rel' for i in range(1, 9)})
df_master = df_master.rename(columns=rename_ge).rename(columns={'ISO3': 'iso3'})

cap_cols = [c for c in df_master.columns if 'mhi_TH' in c or 'mhc_ge' in c]
for col in cap_cols:
    if col in df_master.columns and 'abs' in col:
        df_master[col] = np.minimum(df_master[col].fillna(0), df_master['u18_pop'].fillna(0))

mask = df_master['iso3'].str.upper().isin(['PSE', 'NIC'])
df_master = df_master[~mask]

abs_cols = [c for c in df_master.columns if ('abs' in c and 'norm' not in c) or c in ['total_pop', 'u18_pop']]
for col in abs_cols:
    df_master[col] = pd.to_numeric(df_master[col], errors='coerce').round(0).astype('Int64')

float_cols = df_master.select_dtypes(include='float').columns
df_master[float_cols] = df_master[float_cols].round(2)

# ==========================================
# 7. EXPORT
# ==========================================
all_desired_cols = [
    'iso3', 'adm_name', 'total_pop', 'u18_pop', 'wb_income', 'unicef_ro', 'ucode', 'uuid', 'geometry', 'type', 'fragile',
    'rfl_abs', 'rfl_rel', 'rfl_abs_norm', 'rfl_rel_norm', 'cfl_abs', 'cfl_rel', 'cfl_abs_norm', 'cfl_rel_norm',
    'ts_abs', 'ts_rel', 'ts_abs_norm', 'ts_rel_norm', 'agdr_abs', 'agdr_rel', 'agdr_abs_norm', 'agdr_rel_norm',
    'metdr_spei_abs', 'metdr_spei_rel', 'metdr_spei_abs_norm', 'metdr_spei_rel_norm', 'metdr_spi_abs', 'metdr_spi_rel', 'metdr_spi_abs_norm', 'metdr_spi_rel_norm',
    'hw_fre_abs', 'hw_fre_rel', 'hw_fre_abs_norm', 'hw_fre_rel_norm', 'hw_dur_abs', 'hw_dur_rel', 'hw_dur_abs_norm', 'hw_dur_rel_norm',
    'hw_sev_abs', 'hw_sev_rel', 'hw_sev_abs_norm', 'hw_sev_rel_norm', 'ext_abs', 'ext_rel', 'ext_abs_norm', 'ext_rel_norm',
    'fr_fre_abs', 'fr_fre_rel', 'fr_fre_abs_norm', 'fr_fre_rel_norm', 'fr_int_abs', 'fr_int_rel', 'fr_int_abs_norm', 'fr_int_rel_norm',
    'sds_abs', 'sds_rel', 'sds_abs_norm', 'sds_rel_norm', 'pm25_abs', 'pm25_rel', 'pm25_abs_norm', 'pm25_rel_norm',
    'mal_pv_abs', 'mal_pv_rel', 'mal_pv_abs_norm', 'mal_pv_rel_norm', 'mal_pf_abs', 'mal_pf_rel', 'mal_pf_abs_norm', 'mal_pf_rel_norm',
    'hea_dtp1', 'hea_dtp1_norm', 'hea_dtp3', 'hea_dtp3_norm', 'hea_skat', 'hea_skat_norm', 'hea_elec', 'hea_elec_norm',
    'nut_stu', 'nut_stu_norm', 'nut_fpov', 'nut_fpov_norm', 'wash_wat', 'wash_wat_norm', 'wash_san', 'wash_san_norm', 'wash_hyg', 'wash_hyg_norm',
    'edu_lsos', 'edu_lsos_norm', 'edu_lscr', 'edu_lscr_norm', 'edu_lpov', 'edu_lpov_norm', 'pro_lab', 'pro_lab_norm', 'pro_mar', 'pro_mar_norm',
    'pov_md', 'pov_md_norm', 'pov_u5sp', 'pov_u5sp_norm', 'sur_u5mor', 'sur_u5mor_norm',
    'P1_rfl', 'P1_cfl', 'P1_ts', 'P1_dr', 'P1_hw', 'P1_ext', 'P1_fr', 'P1_sds', 'P1_pm25', 'P1_mal',
    'P2_hea', 'P2_nut', 'P2_wash', 'P2_edu', 'P2_pro', 'P2_pov', 'P2_sur',
    'P1_geometric_avg', 'P2_arithmetic_avg', 'P2_missing_val', 'ccri',
    'mhi_TH75_abs', 'mhi_TH80_abs', 'mhi_TH85_abs', 'mhi_TH90_abs', 'mhi_TH95_abs',
    'mhc_ge1_abs', 'mhc_ge2_abs', 'mhc_ge3_abs', 'mhc_ge4_abs', 'mhc_ge5_abs', 'mhc_ge6_abs', 'mhc_ge7_abs', 'mhc_ge8_abs',
    'mhi_TH75_rel', 'mhi_TH80_rel', 'mhi_TH85_rel', 'mhi_TH90_rel', 'mhi_TH95_rel',
    'mhc_ge1_rel', 'mhc_ge2_rel', 'mhc_ge3_rel', 'mhc_ge4_rel', 'mhc_ge5_rel', 'mhc_ge6_rel', 'mhc_ge7_rel', 'mhc_ge8_rel',
    'drought_topic_abs', 'drought_topic_rel', 'heatwave_topic_abs', 'heatwave_topic_rel', 'fire_topic_abs', 'fire_topic_rel',
    'CCRI_Quadrant'
]
# Final check to ensure columns exist before filtering
final_cols = [c for c in all_desired_cols if c in df_master.columns]
df_master = df_master[final_cols]

print(f"Final row count: {len(df_master)}")
print(f"P2_arithmetic_avg included: {'P2_arithmetic_avg' in df_master.columns}")
print("Exporting to GeoJSON...")
gdf = gpd.GeoDataFrame(df_master, geometry='geometry', crs='EPSG:4326')
output_path = f'{data_dir}/misc/CCRI_P1_P2_format_all_countries.geojson'
gdf.to_file(output_path)
print(f"Complete! Saved to: {output_path}")

Working Directory: /content/drive/MyDrive/CCRI/CCRR_repo/data
Loading base attributes and geometries...
Processing Pillar scores...
Processing Hazard Exposures...
Processing P2 Vulnerabilities...
Final clean up and formatting...
Final row count: 260
P2_arithmetic_avg included: True
Exporting to GeoJSON...
Complete! Saved to: /content/drive/MyDrive/CCRI/CCRR_repo/data/misc/CCRI_P1_P2_format_all_countries.geojson


In [11]:
list(df_master.columns)


['iso3',
 'adm_name',
 'total_pop',
 'u18_pop',
 'wb_income',
 'unicef_ro',
 'ucode',
 'uuid',
 'geometry',
 'type',
 'fragile',
 'rfl_abs',
 'rfl_rel',
 'rfl_abs_norm',
 'rfl_rel_norm',
 'cfl_abs',
 'cfl_rel',
 'cfl_abs_norm',
 'cfl_rel_norm',
 'ts_abs',
 'ts_rel',
 'ts_abs_norm',
 'ts_rel_norm',
 'agdr_abs',
 'agdr_rel',
 'agdr_abs_norm',
 'agdr_rel_norm',
 'metdr_spei_abs',
 'metdr_spei_rel',
 'metdr_spei_abs_norm',
 'metdr_spei_rel_norm',
 'metdr_spi_abs',
 'metdr_spi_rel',
 'metdr_spi_abs_norm',
 'metdr_spi_rel_norm',
 'hw_fre_abs',
 'hw_fre_rel',
 'hw_fre_abs_norm',
 'hw_fre_rel_norm',
 'hw_dur_abs',
 'hw_dur_rel',
 'hw_dur_abs_norm',
 'hw_dur_rel_norm',
 'hw_sev_abs',
 'hw_sev_rel',
 'hw_sev_abs_norm',
 'hw_sev_rel_norm',
 'ext_abs',
 'ext_rel',
 'ext_abs_norm',
 'ext_rel_norm',
 'fr_fre_abs',
 'fr_fre_rel',
 'fr_fre_abs_norm',
 'fr_fre_rel_norm',
 'fr_int_abs',
 'fr_int_rel',
 'fr_int_abs_norm',
 'fr_int_rel_norm',
 'sds_abs',
 'sds_rel',
 'sds_abs_norm',
 'sds_rel_norm',
 'pm2